# mpstab vs qibojit vs PauliPropagation.jl

Explore how the HSMPO surrogate compares, in time and accuracy, with

- **qibojit** (qibo's numba backend): exact statevector, or exact density matrix
  with noise. This is the reference every error is measured against.
- **PauliPropagation.jl**: Heisenberg-picture Pauli-sum propagation, truncated
  at `min_abs_coeff`.

in two scenarios: **exact** (noiseless) and **noisy** (depolarizing noise after
every gate plus ZZ crosstalk). All the plumbing lives in
[`frameworks.py`](frameworks.py); every simulator gets the same qibo circuit,
the same `mpstab.noise` events and the same observable.

**Launch** from the repository root:

```bash
uv run --with juliacall --with qibojit --with pandas --with jupyterlab \
    jupyter lab benchmarks/frameworks.ipynb
```

The first run downloads Julia and PauliPropagation.jl (a few minutes).

**Reading the timings**

- Times are wall-clock for one evaluation. mpstab: everything (Clifford/magic
  split, dressing, MPS evolution, sampling). qibojit: circuit execution and the
  expectation contraction (numba is warmed up first; building the Hamiltonian
  is not timed). PauliPropagation: the propagation, best of two runs so that
  Julia's per-circuit compilation is excluded (translating the circuit is not
  timed).
- Heavy settings grow fast: a 12-qubit density matrix takes ~30 s with
  qibojit, and noiseless PauliPropagation at small `min_abs_coeff` can take
  minutes past depth ~6. Increase sizes gradually.
- qibojit runs multithreaded (`THREADS` below); mpstab and PauliPropagation
  run on one core apart from BLAS.
- Noiseless mpstab is exact up to MPS truncation (`MAX_BOND`). Noisy mpstab is
  exact for zero and one fault and Monte Carlo beyond, so its error should sit
  around its `stderr` (dashed line). PauliPropagation is biased by its
  truncation: lower `MIN_ABS_COEFF` to trade time for accuracy.
- Both mpstab and PauliPropagation exploit the observable's light cone, so
  local observables on shallow circuits are their best case.


In [ ]:
import numpy as np
import pandas as pd
from qibo import set_backend

import frameworks as fw

set_backend("numpy")  # mpstab's own qibo calls; qibojit is used as an object
pd.set_option("display.precision", 4)

## Configuration

Edit and re-run from here.

In [ ]:
THREADS = None  # qibojit threads; None = all cores

# Circuit family: fw.kicked_ising(n, depth) or fw.hardware_efficient(n, depth)
CIRCUIT = fw.kicked_ising

# Observable: a function of n returning a Pauli string or {pauli: coefficient}
OBSERVABLE = fw.local_z  # Z_0 Z_{n-1};  try fw.zz_chain for sum_i Z_i Z_i+1

# Noise: total depolarizing probability per qubit per gate, ZZ crosstalk angle
P_DEPOLARIZING, CROSSTALK = 0.005, 0.02

MPSTAB = dict(max_bond=None, light_cone=True)  # max_bond=None: no truncation
MPSTAB_NOISY = dict(MPSTAB, n_samples=1000, seed=0)
PP = dict(min_abs_coeff=1e-6)  # PauliPropagation truncation; lower = slower, more exact

In [ ]:
jit = fw.qibojit_backend(THREADS)
pp = fw.PauliPropagation()  # installs + warms up Julia on first use
options = dict(qibojit=jit, pauli_propagation=pp, pp_options=PP)

## Sanity check

One small case, every simulator side by side. All three should agree to
machine precision without noise; with noise PauliPropagation should still
match qibojit (up to `MIN_ABS_COEFF`) and mpstab within a few `stderr`.

In [ ]:
n, depth = 8, 3
circuit = CIRCUIT(n, depth)
for events in ([], fw.noise_events(circuit, P_DEPOLARIZING, CROSSTALK)):
    rows = fw.compare(
        circuit, events, OBSERVABLE(n),
        mpstab_options=MPSTAB_NOISY if events else MPSTAB, **options,
    )
    print("noisy" if events else "exact")
    display(pd.DataFrame(rows).set_index("framework")[["value", "error", "time"]])

## Exact simulation

### Width

At fixed depth the statevector doubles with every qubit, while mpstab and
PauliPropagation only see the observable's light cone.

In [ ]:
DEPTH = 3
SIZES = [8, 12, 16, 20, 22, 24]

exact_width = fw.sweep(
    ((n, CIRCUIT(n, DEPTH), [], OBSERVABLE(n)) for n in SIZES),
    label="n", mpstab_options=MPSTAB, **options,
)
fw.plot(exact_width, "n", f"exact, depth {DEPTH}");

In [ ]:
exact_width.pivot(index="n", columns="framework", values=["time", "error"])

### Depth

At fixed width the light cone fills up with depth: the MPS bond dimension
grows for mpstab, the number of Pauli terms for PauliPropagation.

In [ ]:
N = 12
DEPTHS = [1, 2, 3, 4, 6]  # noiseless PauliPropagation blows up past ~6 here

exact_depth = fw.sweep(
    ((d, CIRCUIT(N, d), [], OBSERVABLE(N)) for d in DEPTHS),
    label="depth", mpstab_options=MPSTAB, **options,
)
fw.plot(exact_depth, "depth", f"exact, {N} qubits");

In [ ]:
exact_depth.pivot(index="depth", columns="framework", values=["time", "bond", "terms"])

## Noisy simulation

### Width

qibojit now evolves a density matrix (4^n entries), so it stops at
`fw.QIBOJIT_MAX_QUBITS["density_matrix"]` qubits; beyond that there is no
reference and only the times are shown.

In [ ]:
DEPTH = 2
SIZES = [6, 8, 10, 11, 12, 16, 20]

def noisy_case(n, depth):
    circuit = CIRCUIT(n, depth)
    return circuit, fw.noise_events(circuit, P_DEPOLARIZING, CROSSTALK), OBSERVABLE(n)

noisy_width = fw.sweep(
    ((n, *noisy_case(n, DEPTH)) for n in SIZES),
    label="n", mpstab_options=MPSTAB_NOISY, **options,
)
fw.plot(noisy_width, "n", f"noisy, depth {DEPTH}, p={P_DEPOLARIZING}, ZZ {CROSSTALK}");

### Depth

More gates mean more fault locations: the probability of two or more faults
grows, and with it mpstab's sampling error at fixed `n_samples`.

In [ ]:
N = 10
DEPTHS = [1, 2, 3, 4, 6]

noisy_depth = fw.sweep(
    ((d, *noisy_case(N, d)) for d in DEPTHS),
    label="depth", mpstab_options=MPSTAB_NOISY, **options,
)
fw.plot(noisy_depth, "depth", f"noisy, {N} qubits");

In [ ]:
noisy_depth[noisy_depth.framework == "mpstab"][
    ["depth", "value", "error", "stderr", "p_two_or_more", "patterns", "time"]
]

### Accuracy vs cost

The fair comparison for sampled and truncated methods: how much time does each
need to reach a given error? Sweep mpstab's `n_samples` and
PauliPropagation's `min_abs_coeff` on one case.

In [ ]:
n, depth = 10, 4
circuit, events, observable = noisy_case(n, depth)
reference = fw.run_qibojit(jit, circuit, events, observable)["value"]

rows = []
for n_samples in [100, 300, 1000, 3000]:
    r = fw.run_mpstab_noisy(circuit, events, observable, **dict(MPSTAB_NOISY, n_samples=n_samples))
    rows.append(dict(framework="mpstab", setting=n_samples, time=r["time"],
                     error=abs(r["value"] - reference), stderr=r["stderr"]))
for threshold in [1e-2, 1e-3, 1e-4, 1e-5, 1e-6]:
    r = pp.run(circuit, events, observable, min_abs_coeff=threshold)
    rows.append(dict(framework="PauliPropagation", setting=threshold, time=r["time"],
                     error=abs(r["value"] - reference), terms=r["terms"]))
tradeoff = pd.DataFrame(rows)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
for name, group in tradeoff.groupby("framework", sort=False):
    ax.plot(group["time"], group["error"].clip(lower=1e-17), "o-",
            color=fw.COLORS[name], linewidth=2, label=name)
    for _, row in group.iterrows():
        ax.annotate(f"{row['setting']:g}", (row["time"], max(row["error"], 1e-17)),
                    textcoords="offset points", xytext=(4, 4), fontsize=8)
ax.set(xscale="log", yscale="log", xlabel="wall time [s]", ylabel="|value - qibojit|",
       title=f"noisy, {n} qubits, depth {depth}: labels are n_samples / min_abs_coeff")
ax.grid(alpha=0.3); ax.legend(frameon=False)
tradeoff

## Playground

Any circuit and observable: build the events with `fw.noise_events` (or by
hand with `mpstab.noise.PauliChannel` / `CoherentRotation`) and pass them to
`fw.compare`.

In [ ]:
circuit = fw.hardware_efficient(12, 4, seed=1)
events = fw.noise_events(circuit, p=0.002, crosstalk=0.0)
observable = {"ZZ" + "I" * 10: 1.0, "I" * 5 + "XX" + "I" * 5: -0.5}
pd.DataFrame(fw.compare(circuit, events, observable, mpstab_options=MPSTAB_NOISY, **options))